In [1]:

import os
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python environment initialized")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("Running on CPU. We will not start model training yet.")


Python environment initialized
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:

from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = "/content/drive/MyDrive/BharatDocAI"

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments/finetuning"
)

METRICS_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs/metrics/finetuning"
)

BASE_CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "models/checkpoints/qwen3_lora"
)

EXPERIMENT2_CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "models/checkpoints/qwen3_lora_experiment2"
)

TOKENIZED_DATA_PATH = os.path.join(
    OUTPUT_DIR,
    "experiment2_tokenized_training.json"
)

os.makedirs(EXPERIMENT2_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

print("Google Drive mounted successfully")
print("Project directory:", PROJECT_ROOT)
print("Experiment 2 checkpoint:", EXPERIMENT2_CHECKPOINT_DIR)


Mounted at /content/drive
Google Drive mounted successfully
Project directory: /content/drive/MyDrive/BharatDocAI
Experiment 2 checkpoint: /content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora_experiment2


In [3]:

required_files = {
    "Experiment 2 tokenized training data": TOKENIZED_DATA_PATH,
    "Experiment 1 LoRA adapter": os.path.join(
        BASE_CHECKPOINT_DIR,
        "adapter_model.safetensors"
    )
}

for name, path in required_files.items():
    print(
        f"{'FOUND' if os.path.exists(path) else 'MISSING'}: {name}"
    )
    print(path)

assert os.path.exists(TOKENIZED_DATA_PATH), (
    "Experiment 2 training data not found!"
)

with open(TOKENIZED_DATA_PATH, "r", encoding="utf-8") as f:
    experiment2_tokenized_train = json.load(f)

print("\nTraining examples loaded:", len(experiment2_tokenized_train))

print(
    "First example keys:",
    list(experiment2_tokenized_train[0].keys())

    )


FOUND: Experiment 2 tokenized training data
/content/drive/MyDrive/BharatDocAI/experiments/finetuning/experiment2_tokenized_training.json
FOUND: Experiment 1 LoRA adapter
/content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora/adapter_model.safetensors

Training examples loaded: 162
First example keys: ['input_ids', 'attention_mask', 'labels', 'supported_fields', 'masked_fields', 'target_json', 'document_id', 'ocr_engine', 'language']


In [5]:

%pip install -q bitsandbytes==0.50.2


In [6]:

import bitsandbytes as bnb
import transformers
import peft
import accelerate

print("BitsAndBytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

print("CUDA available:", torch.cuda.is_available())


BitsAndBytes: 0.50.2
Transformers: 5.22.1
PEFT: 0.21.2
Accelerate: 1.15.0
CUDA available: True


In [7]:

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import torch.nn as nn
import torch.optim as optim

import transformers
import peft
import bitsandbytes as bnb

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bnb.__version__)
print("All required dependencies imported successfully.")


Transformers: 5.22.1
PEFT: 0.21.2
BitsAndBytes: 0.50.2
All required dependencies imported successfully.


In [8]:

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", MODEL_NAME)
print("Vocabulary size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Fast tokenizer:", tokenizer.is_fast)

assert tokenizer.is_fast


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Tokenizer loaded: Qwen/Qwen3-4B
Vocabulary size: 151669
Pad token: <|endoftext|>
EOS token: <|im_end|>
Fast tokenizer: True


In [9]:

class Experiment2Dataset(Dataset):

    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def experiment2_collate_fn(batch):

    input_ids = [
        torch.tensor(x["input_ids"], dtype=torch.long)
        for x in batch
    ]

    attention_masks = [
        torch.tensor(x["attention_mask"], dtype=torch.long)
        for x in batch
    ]

    labels = [
        torch.tensor(x["labels"], dtype=torch.long)
        for x in batch
    ]

    return {
        "input_ids": pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=tokenizer.pad_token_id
        ),
        "attention_mask": pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        ),
        "labels": pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100
        )
    }


experiment2_dataset = Experiment2Dataset(
    experiment2_tokenized_train
)

BATCH_SIZE = 2

experiment2_train_loader = DataLoader(
    experiment2_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=experiment2_collate_fn
)

print("Training examples:", len(experiment2_dataset))
print("Training batches:", len(experiment2_train_loader))

sample_batch = next(iter(experiment2_train_loader))

for key, value in sample_batch.items():
    print(key, value.shape)

print(
    "Supervised tokens:",
    (sample_batch["labels"] != -100).sum().item()
)


Training examples: 162
Training batches: 81
input_ids torch.Size([2, 286])
attention_mask torch.Size([2, 286])
labels torch.Size([2, 286])
Supervised tokens: 115


In [10]:

UNIFIED_DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "experiments/baselines/llm_extraction/unified_extraction_dataset.jsonl"
)

with open(UNIFIED_DATASET_PATH, "r", encoding="utf-8") as f:
    unified_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

validation_records = [
    record for record in unified_records
    if record["split"] == "validation"
    and record["corruption"] == "clean"
]

print("Unified records:", len(unified_records))
print("Clean validation records:", len(validation_records))
print(
    "Unique validation documents:",
    len(set(r["document_id"] for r in validation_records))
)

print("\nLanguage distribution:")
display(
    pd.Series(
        [r["language"] for r in validation_records]
    ).value_counts()
)

assert len(validation_records) == 54
assert len(set(r["document_id"] for r in validation_records)) == 18

print("\nValidation data loaded successfully.")


Unified records: 4320
Clean validation records: 54
Unique validation documents: 18

Language distribution:


,count
english,18
hindi,18
kannada,18



Validation data loaded successfully.


In [11]:

import re
import unicodedata

CATEGORY_SCHEMAS = {
    "invoice": [
        "name", "date", "document_id", "address", "amount"
    ],
    "application": [
        "name", "date", "document_id", "address", "phone"
    ],
    "certificate": [
        "name", "date", "document_id", "institution", "grade"
    ]
}

SYSTEM_PROMPT = """
You are an information extraction system.
Extract only information explicitly supported by the OCR text.
Never guess or invent missing values.
Return exactly the requested fields as valid JSON.
Use an empty string when a field is missing or unreadable.
"""

def normalize_evidence(value, field):
    value = unicodedata.normalize("NFC", str(value))
    value = value.replace("\u200c", "").replace("\u200d", "")
    value = value.strip().casefold()

    if field in ["date", "phone", "amount", "document_id"]:
        value = re.sub(r"[\s/\\\-:.,|]", "", value)
    else:
        value = re.sub(r"\s+", " ", value)

    return value


# Construct validation-only evidence keys
validation_exact_keys = set()

for record in validation_records:
    for field in CATEGORY_SCHEMAS[record["category"]]:

        expected = str(
            record["ground_truth"].get(field, "")
        ).strip()

        ocr_text = record["ocr_text"]

        if (
            expected
            and normalize_evidence(expected, field)
            in normalize_evidence(ocr_text, field)
        ):
            validation_exact_keys.add((
                record["document_id"],
                record["ocr_engine"],
                field
            ))


print("Validation exact-supported fields:", len(validation_exact_keys))
print("Total validation fields:", sum(
    len(CATEGORY_SCHEMAS[r["category"]])
    for r in validation_records
))


Validation exact-supported fields: 208
Total validation fields: 270


In [15]:
MAX_LENGTH = 1024

print("Maximum sequence length:", MAX_LENGTH)

Maximum sequence length: 1024


In [16]:

def create_evidence_aware_example(record, support_keys):

    fields = CATEGORY_SCHEMAS[record["category"]]

    target = {}
    supported_fields = []
    masked_fields = []

    for field in fields:

        key = (
            record["document_id"],
            record["ocr_engine"],
            field
        )

        expected = str(
            record["ground_truth"].get(field, "")
        ).strip()

        if key in support_keys:
            target[field] = expected
            supported_fields.append(field)
        else:
            target[field] = ""
            masked_fields.append(field)

    target_json = json.dumps(
        target,
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": (
                f"Language: {record['language']}\n"
                f"Category: {record['category']}\n"
                f"OCR Text:\n{record['ocr_text']}\n\n"
                f"Extract these fields: {', '.join(fields)}"
            )
        },
        {
            "role": "assistant",
            "content": target_json
        }
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    target_start = full_text.rfind(target_json)

    if target_start == -1:
        raise ValueError(
            f"Target not found: {record['document_id']}"
        )

    labels = [-100] * len(input_ids)

    masked_spans = []

    for field in masked_fields:

        property_text = (
            json.dumps(field, ensure_ascii=False) + ':""'
        )

        position = target_json.find(property_text)

        if position == -1:
            raise ValueError(
                f"Property not found: {field}"
            )

        start = target_start + position
        end = start + len(property_text)

        masked_spans.append((start, end))

    for i, (start, end) in enumerate(offsets):

        if end <= start or start < target_start:
            continue

        overlaps_masked = any(
            start < masked_end and end > masked_start
            for masked_start, masked_end in masked_spans
        )

        if not overlaps_masked:
            labels[i] = input_ids[i]

    if len(input_ids) > MAX_LENGTH:
        raise ValueError(
            f"Sequence too long: {record['document_id']}"
        )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "supported_fields": supported_fields,
        "masked_fields": masked_fields,
        "target_json": target_json,
        "document_id": record["document_id"],
        "ocr_engine": record["ocr_engine"],
        "language": record["language"]
    }


experiment2_tokenized_validation = [
    create_evidence_aware_example(
        record,
        validation_exact_keys
    )
    for record in validation_records
]

print("Validation examples:", len(experiment2_tokenized_validation))

print(
    "Supported fields:",
    sum(len(x["supported_fields"])
        for x in experiment2_tokenized_validation)
)

print(
    "Masked fields:",
    sum(len(x["masked_fields"])
        for x in experiment2_tokenized_validation)
)

print(
    "Supervised tokens:",
    sum(
        sum(label != -100 for label in x["labels"])
        for x in experiment2_tokenized_validation
    )
)


Validation examples: 54
Supported fields: 208
Masked fields: 62
Supervised tokens: 2538


In [17]:

experiment2_validation_dataset = Experiment2Dataset(
    experiment2_tokenized_validation
)

experiment2_validation_loader = DataLoader(
    experiment2_validation_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=experiment2_collate_fn
)

print("Validation examples:", len(experiment2_validation_dataset))
print("Validation batches:", len(experiment2_validation_loader))

validation_batch = next(iter(experiment2_validation_loader))

for key, value in validation_batch.items():
    print(key, value.shape)

print(
    "Validation supervised tokens:",
    (validation_batch["labels"] != -100).sum().item()
)


Validation examples: 54
Validation batches: 27
input_ids torch.Size([2, 186])
attention_mask torch.Size([2, 186])
labels torch.Size([2, 186])
Validation supervised tokens: 92


In [18]:

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit NF4 quantization configured.")
print("Compute dtype:", bnb_config.bnb_4bit_compute_dtype)
print("Double quantization:", bnb_config.bnb_4bit_use_double_quant)


4-bit NF4 quantization configured.
Compute dtype: torch.float16
Double quantization: True


In [19]:

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory before loading:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

print("\nBase model loaded successfully.")
print("Model:", MODEL_NAME)
print("Device:", model.get_input_embeddings().weight.device)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


GPU memory before loading: 0.0 GB


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Base model loaded successfully.
Model: Qwen/Qwen3-4B
Device: cuda:0
GPU memory allocated: 3.22 GB


In [20]:

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTrainable parameters:", trainable_params)
print("Gradient checkpointing:", model.is_gradient_checkpointing)


trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089

Trainable parameters: 16515072
Gradient checkpointing: True


In [21]:

EPOCHS = 5
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.3

optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = (
    len(experiment2_train_loader)
    + GRADIENT_ACCUMULATION_STEPS - 1
) // GRADIENT_ACCUMULATION_STEPS

total_training_steps = steps_per_epoch * EPOCHS
warmup_steps = 5

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Training configuration:")
print("Epochs:", EPOCHS)
print("Batches per epoch:", len(experiment2_train_loader))
print("Optimizer steps per epoch:", steps_per_epoch)
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Learning rate:", LEARNING_RATE)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)


Training configuration:
Epochs: 5
Batches per epoch: 81
Optimizer steps per epoch: 21
Total optimizer steps: 105
Warmup steps: 5
Learning rate: 0.0002
Effective batch size: 8


In [22]:

from tqdm.auto import tqdm

# Reproducibility
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Training settings
best_validation_loss = float("inf")
best_epoch = 0
training_history = []

# FP16 gradient scaler
scaler = torch.amp.GradScaler("cuda")

# Save configuration
EXPERIMENT2_CONFIG = {
    "model": MODEL_NAME,
    "experiment": "Experiment 2: Evidence-Aware Supervision",
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_rank": 8,
    "lora_alpha": 16,
    "max_length": MAX_LENGTH,
    "training_examples": len(experiment2_dataset),
    "validation_examples": len(experiment2_validation_dataset),
    "supported_training_fields": 621,
    "masked_training_fields": 189
}

config_path = os.path.join(
    EXPERIMENT2_CHECKPOINT_DIR,
    "experiment2_config.json"
)

with open(config_path, "w") as f:
    json.dump(EXPERIMENT2_CONFIG, f, indent=4)


def move_batch_to_gpu(batch):
    return {
        key: value.to("cuda")
        for key, value in batch.items()
    }


def calculate_validation_loss():

    model.eval()

    total_loss = 0.0
    total_supervised_tokens = 0

    with torch.no_grad():

        for batch in experiment2_validation_loader:

            batch = move_batch_to_gpu(batch)

            supervised_tokens = (
                batch["labels"] != -100
            ).sum().item()

            if supervised_tokens == 0:
                continue

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

            total_loss += (
                outputs.loss.item() * supervised_tokens
            )

            total_supervised_tokens += supervised_tokens

    if total_supervised_tokens == 0:
        raise ValueError("No supervised validation tokens.")

    return total_loss / total_supervised_tokens


print("Starting Experiment 2 QLoRA training...")
print("Device: cuda:0")
print("Total optimizer steps:", total_training_steps)
print("Warmup steps:", warmup_steps)
print("Checkpoint directory:", EXPERIMENT2_CHECKPOINT_DIR)

for epoch in range(EPOCHS):

    model.train()

    epoch_loss_sum = 0.0
    epoch_token_count = 0

    # Materialize batches to handle the final partial accumulation group
    epoch_batches = list(experiment2_train_loader)

    progress_bar = tqdm(
        range(0, len(epoch_batches), GRADIENT_ACCUMULATION_STEPS),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for group_start in progress_bar:

        group = epoch_batches[
            group_start:
            group_start + GRADIENT_ACCUMULATION_STEPS
        ]

        # Count all supervised tokens in this accumulation group
        group_token_count = sum(
            (batch["labels"] != -100).sum().item()
            for batch in group
        )

        if group_token_count == 0:
            continue

        optimizer.zero_grad(set_to_none=True)

        for batch in group:

            batch = move_batch_to_gpu(batch)

            supervised_tokens = (
                batch["labels"] != -100
            ).sum().item()

            if supervised_tokens == 0:
                continue

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

                # Token-weighted gradient accumulation
                weighted_loss = (
                    outputs.loss
                    * supervised_tokens
                    / group_token_count
                )

            scaler.scale(weighted_loss).backward()

            epoch_loss_sum += (
                outputs.loss.detach().item()
                * supervised_tokens
            )

            epoch_token_count += supervised_tokens

        # Update once per accumulation group
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        progress_bar.set_postfix({
            "train_loss": round(
                epoch_loss_sum / max(epoch_token_count, 1),
                5
            )
        })

    train_loss = epoch_loss_sum / epoch_token_count

    # Validation
    validation_loss = calculate_validation_loss()

    current_lr = scheduler.get_last_lr()[0]

    epoch_result = {
        "epoch": epoch + 1,
        "training_loss": train_loss,
        "validation_loss": validation_loss,
        "learning_rate": current_lr,
        "supervised_training_tokens": epoch_token_count
    }

    training_history.append(epoch_result)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training loss:   {train_loss:.5f}")
    print(f"Validation loss: {validation_loss:.5f}")
    print(f"Learning rate:   {current_lr:.8f}")

    # Save the best adapter
    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss
        best_epoch = epoch + 1

        model.save_pretrained(
            EXPERIMENT2_CHECKPOINT_DIR
        )

        tokenizer.save_pretrained(
            EXPERIMENT2_CHECKPOINT_DIR
        )

        print("Best Experiment 2 checkpoint saved!")

    # Save training history after every epoch
    history_path = os.path.join(
        METRICS_DIR,
        "experiment2_training_history.json"
    )

    with open(history_path, "w") as f:
        json.dump(training_history, f, indent=4)

    gc.collect()
    torch.cuda.empty_cache()


print("\nExperiment 2 training completed!")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_validation_loss, 5))
print("Checkpoint:", EXPERIMENT2_CHECKPOINT_DIR)


Starting Experiment 2 QLoRA training...
Device: cuda:0
Total optimizer steps: 105
Warmup steps: 5
Checkpoint directory: /content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora_experiment2


Epoch 1/5:   0%|          | 0/21 [00:00<?, ?it/s]


Epoch 1/5
Training loss:   0.10203
Validation loss: 0.00221
Learning rate:   0.00016800
Best Experiment 2 checkpoint saved!


Epoch 2/5:   0%|          | 0/21 [00:00<?, ?it/s]


Epoch 2/5
Training loss:   0.00167
Validation loss: 0.00034
Learning rate:   0.00012600
Best Experiment 2 checkpoint saved!


Epoch 3/5:   0%|          | 0/21 [00:00<?, ?it/s]


Epoch 3/5
Training loss:   0.00091
Validation loss: 0.00035
Learning rate:   0.00008400


Epoch 4/5:   0%|          | 0/21 [00:00<?, ?it/s]


Epoch 4/5
Training loss:   0.00032
Validation loss: 0.00030
Learning rate:   0.00004200
Best Experiment 2 checkpoint saved!


Epoch 5/5:   0%|          | 0/21 [00:00<?, ?it/s]


Epoch 5/5
Training loss:   0.00019
Validation loss: 0.00028
Learning rate:   0.00000000
Best Experiment 2 checkpoint saved!

Experiment 2 training completed!
Best epoch: 5
Best validation loss: 0.00028
Checkpoint: /content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora_experiment2


In [23]:

# ==========================================
# EXPERIMENT 2: TEST SET PREPARATION
# ==========================================

test_records = [
    record for record in unified_records
    if record["split"] == "test"
    and record["corruption"] == "clean"
]

print("Total clean test records:", len(test_records))

print(
    "Unique test documents:",
    len(set(r["document_id"] for r in test_records))
)

print("\nLanguage distribution:")
display(
    pd.Series(
        [r["language"] for r in test_records]
    ).value_counts()
)

print("\nOCR engine distribution:")
display(
    pd.Series(
        [r["ocr_engine"] for r in test_records]
    ).value_counts()
)

print("\nDocument category distribution:")
display(
    pd.Series(
        [r["category"] for r in test_records]
    ).value_counts()
)

# Integrity checks
assert len(test_records) == 54
assert len(set(r["document_id"] for r in test_records)) == 18

assert set(r["split"] for r in test_records) == {"test"}

assert set(r["corruption"] for r in test_records) == {"clean"}

print("\nTest set integrity checks passed!")


Total clean test records: 54
Unique test documents: 18

Language distribution:


,count
english,18
hindi,18
kannada,18



OCR engine distribution:


,count
tesseract,18
easyocr,18
paddleocr,18



Document category distribution:


,count
invoice,18
application,18
certificate,18



Test set integrity checks passed!


In [24]:

import os

adapter_path = os.path.join(
    EXPERIMENT2_CHECKPOINT_DIR,
    "adapter_model.safetensors"
)

adapter_config_path = os.path.join(
    EXPERIMENT2_CHECKPOINT_DIR,
    "adapter_config.json"
)

print("Adapter weights:", os.path.exists(adapter_path))
print("Adapter config:", os.path.exists(adapter_config_path))

if os.path.exists(adapter_path):
    print(
        "Adapter size:",
        round(os.path.getsize(adapter_path) / (1024**2), 2),
        "MB"
    )

assert os.path.exists(adapter_path)
assert os.path.exists(adapter_config_path)

print("\nExperiment 2 checkpoint verified!")


Adapter weights: True
Adapter config: True
Adapter size: 63.06 MB

Experiment 2 checkpoint verified!


In [25]:

# ==========================================
# EXPERIMENT 2: TEST SET INFERENCE
# ==========================================

import os
import json
import re
import torch
import pandas as pd
from tqdm.auto import tqdm

# Confirm the trained model is still available
assert "model" in globals(), "Model not loaded. Restore the model first."
assert "tokenizer" in globals(), "Tokenizer not loaded."
assert len(test_records) == 54

model.eval()

# Ensure inference uses the trained Experiment 2 adapter
assert hasattr(model, "peft_config"), (
    "The current model does not appear to have a PEFT adapter attached."
)

print("Model ready for inference.")
print("Test records:", len(test_records))
print("Adapter:", EXPERIMENT2_CHECKPOINT_DIR)


# ------------------------------------------
# JSON extraction helper
# ------------------------------------------

def parse_model_json(response):

    try:
        # Remove possible markdown code fences
        cleaned = response.strip()

        cleaned = re.sub(
            r"^```(?:json)?\s*",
            "",
            cleaned,
            flags=re.IGNORECASE
        )

        cleaned = re.sub(
            r"\s*```$",
            "",
            cleaned
        )

        # Extract JSON object
        start = cleaned.find("{")
        end = cleaned.rfind("}")

        if start == -1 or end == -1 or end < start:
            return None

        return json.loads(cleaned[start:end + 1])

    except (json.JSONDecodeError, TypeError):
        return None


# ------------------------------------------
# Inference function
# ------------------------------------------

def generate_experiment2_prediction(record):

    fields = CATEGORY_SCHEMAS[record["category"]]

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": (
                f"Language: {record['language']}\n"
                f"Category: {record['category']}\n"
                f"OCR Text:\n{record['ocr_text']}\n\n"
                f"Extract these fields: {', '.join(fields)}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    input_length = inputs["input_ids"].shape[-1]

    with torch.inference_mode():

        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_model_json(response)

    return response, prediction


# ------------------------------------------
# Run inference on all 54 records
# ------------------------------------------

experiment2_test_results = []

for record in tqdm(
    test_records,
    desc="Experiment 2 Test Inference"
):

    raw_response, prediction = generate_experiment2_prediction(
        record
    )

    experiment2_test_results.append({

        "document_id": record["document_id"],
        "language": record["language"],
        "category": record["category"],
        "ocr_engine": record["ocr_engine"],

        "ocr_text": record["ocr_text"],

        "ground_truth": record["ground_truth"],

        "prediction": prediction,

        "raw_response": raw_response,

        "json_valid": prediction is not None

    })


# ------------------------------------------
# Save results
# ------------------------------------------

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments/finetuning"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

EXPERIMENT2_TEST_RESULTS_PATH = os.path.join(
    RESULTS_DIR,
    "experiment2_test_predictions.jsonl"
)

with open(
    EXPERIMENT2_TEST_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:

    for result in experiment2_test_results:

        f.write(
            json.dumps(
                result,
                ensure_ascii=False
            ) + "\n"
        )


# ------------------------------------------
# Initial inference summary
# ------------------------------------------

valid_json_count = sum(
    result["json_valid"]
    for result in experiment2_test_results
)

print("\nExperiment 2 inference completed!")

print("Total predictions:", len(experiment2_test_results))

print(
    "Valid JSON:",
    valid_json_count,
    "/",
    len(experiment2_test_results)
)

print(
    "Invalid JSON:",
    len(experiment2_test_results) - valid_json_count
)

print(
    "\nSaved predictions to:",
    EXPERIMENT2_TEST_RESULTS_PATH
)

# Display a few examples
for result in experiment2_test_results[:3]:

    print("\nDocument:", result["document_id"])
    print("Language:", result["language"])
    print("OCR engine:", result["ocr_engine"])
    print("Ground truth:", result["ground_truth"])
    print("Prediction:", result["prediction"])


Model ready for inference.
Test records: 54
Adapter: /content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora_experiment2


Experiment 2 Test Inference:   0%|          | 0/54 [00:00<?, ?it/s]


Experiment 2 inference completed!
Total predictions: 54
Valid JSON: 54 / 54
Invalid JSON: 0

Saved predictions to: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/experiment2_test_predictions.jsonl

Document: english_invoice_009
Language: english
OCR engine: tesseract
Ground truth: {'name': 'Ananya Joshi', 'date': '09/01/2026', 'document_id': 'INV-0009', 'address': 'Hassan', 'amount': '2733'}
Prediction: {'name': 'Ananya Joshi', 'date': '09/01/2026', 'document_id': 'INV-0009', 'address': 'Hassan', 'amount': '2733'}

Document: english_invoice_010
Language: english
OCR engine: tesseract
Ground truth: {'name': 'Sneha Singh', 'date': '10/01/2026', 'document_id': 'INV-0010', 'address': 'Ballari', 'amount': '2870'}
Prediction: {'name': 'Sneha Singh', 'date': '10/01/2026', 'document_id': 'INV-0010', 'address': 'Ballari', 'amount': '2870'}

Document: english_application_019
Language: english
OCR engine: tesseract
Ground truth: {'name': 'Sneha Verma', 'date': '19/01/2026', 'document_

In [26]:
import json
import os
import pandas as pd
from collections import defaultdict

PREDICTION_PATH = os.path.join(
    PROJECT_ROOT,
    "experiments/finetuning/experiment2_test_predictions.jsonl"
)

# Load predictions
with open(PREDICTION_PATH, "r", encoding="utf-8") as f:
    predictions = [json.loads(line) for line in f if line.strip()]

# Metric counters
total_fields = 0
raw_correct = 0
normalized_correct = 0
exact_documents = 0
valid_json = 0
schema_valid = 0
missing_predictions = 0

tp = fp = fn = 0
supported_predictions = 0
nonempty_predictions = 0

language_stats = defaultdict(lambda: {"correct": 0, "total": 0, "documents": 0})
engine_stats = defaultdict(lambda: {"correct": 0, "total": 0, "documents": 0})
category_stats = defaultdict(lambda: {"correct": 0, "total": 0, "documents": 0})

for record in predictions:
    gt = record["ground_truth"]
    pred = record["prediction"]
    fields = CATEGORY_SCHEMAS[record["category"]]

    # JSON and schema checks
    if record.get("json_valid", False):
        valid_json += 1

    if isinstance(pred, dict) and set(pred.keys()) == set(fields):
        schema_valid += 1

    document_correct = True

    for field in fields:
        expected = str(gt.get(field, ""))
        actual = str(pred.get(field, ""))

        expected_norm = normalize_evidence(expected, field)
        actual_norm = normalize_evidence(actual, field)

        total_fields += 1

        raw_match = actual == expected
        normalized_match = actual_norm == expected_norm

        if raw_match:
            raw_correct += 1

        if normalized_match:
            normalized_correct += 1
        else:
            document_correct = False

        if actual_norm == "":
            missing_predictions += 1

        # Extraction precision / recall counts.
        # A correct non-empty value is a true positive.
        # An incorrect non-empty value is a false positive.
        # A non-empty expected value not correctly extracted is a false negative.
        if actual_norm != "":
            nonempty_predictions += 1

            if normalized_match and expected_norm != "":
                tp += 1
            else:
                fp += 1

        if expected_norm != "" and not normalized_match:
            fn += 1

        # Exact OCR substring support diagnostic.
        ocr_text = normalize_evidence(record.get("ocr_text", ""), field)

        if actual_norm != "" and actual_norm in ocr_text:
            supported_predictions += 1

        # Group metrics
        for stats, group in [
            (language_stats, record["language"]),
            (engine_stats, record["ocr_engine"]),
            (category_stats, record["category"])
        ]:
            stats[group]["total"] += 1
            if normalized_match:
                stats[group]["correct"] += 1

    if document_correct:
        exact_documents += 1

    language_stats[record["language"]]["documents"] += int(document_correct)
    engine_stats[record["ocr_engine"]]["documents"] += int(document_correct)
    category_stats[record["category"]]["documents"] += int(document_correct)


# Calculate aggregate metrics
field_accuracy_raw = raw_correct / total_fields
field_accuracy_normalized = normalized_correct / total_fields
exact_match_rate = exact_documents / len(predictions)

precision = tp / (tp + fp) if (tp + fp) else 0
recall = tp / (tp + fn) if (tp + fn) else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

ocr_support_rate = (
    supported_predictions / nonempty_predictions
    if nonempty_predictions else 0
)

print("=" * 55)
print("EXPERIMENT 2: TEST SET EVALUATION")
print("=" * 55)

print(f"Documents evaluated:          {len(predictions)}")
print(f"Total fields evaluated:       {total_fields}")
print(f"Valid JSON:                   {valid_json}/{len(predictions)}")
print(f"Schema-valid predictions:     {schema_valid}/{len(predictions)}")
print(f"Raw field accuracy:           {field_accuracy_raw:.2%}")
print(f"Normalized field accuracy:    {field_accuracy_normalized:.2%}")
print(f"Exact document match:         {exact_documents}/{len(predictions)} ({exact_match_rate:.2%})")
print(f"Extraction precision:         {precision:.2%}")
print(f"Extraction recall:            {recall:.2%}")
print(f"Extraction F1:                {f1:.2%}")
print(f"Empty predicted fields:       {missing_predictions}/{total_fields}")
print(f"Non-empty predictions:        {nonempty_predictions}")
print(f"Exact OCR substring support:  {ocr_support_rate:.2%}")

def display_group_metrics(stats, group_name):
    rows = []
    for group, values in sorted(stats.items()):
        rows.append({
            group_name: group,
            "Correct fields": values["correct"],
            "Total fields": values["total"],
            "Field accuracy": values["correct"] / values["total"],
            "Exact documents": values["documents"]
        })

    return pd.DataFrame(rows)

print("\nACCURACY BY LANGUAGE")
display(display_group_metrics(language_stats, "Language"))

print("\nACCURACY BY OCR ENGINE")
display(display_group_metrics(engine_stats, "OCR Engine"))

print("\nACCURACY BY DOCUMENT CATEGORY")
display(display_group_metrics(category_stats, "Category"))

EXPERIMENT 2: TEST SET EVALUATION
Documents evaluated:          54
Total fields evaluated:       270
Valid JSON:                   54/54
Schema-valid predictions:     54/54
Raw field accuracy:           75.19%
Normalized field accuracy:    75.19%
Exact document match:         19/54 (35.19%)
Extraction precision:         75.19%
Extraction recall:            75.19%
Extraction F1:                75.19%
Empty predicted fields:       0/270
Non-empty predictions:        270
Exact OCR substring support:  88.89%

ACCURACY BY LANGUAGE


,Language,Correct fields,Total fields,Field accuracy,Exact documents
0,english,88,90,0.977778,16
1,hindi,69,90,0.766667,2
2,kannada,46,90,0.511111,1



ACCURACY BY OCR ENGINE


,OCR Engine,Correct fields,Total fields,Field accuracy,Exact documents
0,easyocr,74,90,0.822222,7
1,paddleocr,59,90,0.655556,7
2,tesseract,70,90,0.777778,5



ACCURACY BY DOCUMENT CATEGORY


,Category,Correct fields,Total fields,Field accuracy,Exact documents
0,application,70,90,0.777778,8
1,certificate,64,90,0.711111,5
2,invoice,69,90,0.766667,6


In [27]:
from collections import defaultdict
import pandas as pd

field_stats = defaultdict(lambda: {
    "correct": 0,
    "total": 0,
    "unsupported": 0,
    "incorrect_examples": []
})

for record in predictions:
    gt = record["ground_truth"]
    pred = record["prediction"]
    fields = CATEGORY_SCHEMAS[record["category"]]

    for field in fields:
        expected = str(gt.get(field, ""))
        actual = str(pred.get(field, ""))

        expected_norm = normalize_evidence(expected, field)
        actual_norm = normalize_evidence(actual, field)
        ocr_norm = normalize_evidence(record["ocr_text"], field)

        field_stats[field]["total"] += 1

        if expected_norm == actual_norm:
            field_stats[field]["correct"] += 1
        else:
            field_stats[field]["incorrect_examples"].append({
                "document": record["document_id"],
                "language": record["language"],
                "expected": expected,
                "predicted": actual,
                "ocr_text": record["ocr_text"]
            })

        if actual_norm and actual_norm not in ocr_norm:
            field_stats[field]["unsupported"] += 1

rows = []

for field, values in sorted(field_stats.items()):
    rows.append({
        "Field": field,
        "Correct": values["correct"],
        "Total": values["total"],
        "Accuracy": values["correct"] / values["total"],
        "Unsupported": values["unsupported"]
    })

field_df = pd.DataFrame(rows).sort_values("Accuracy")

print("FIELD-LEVEL PERFORMANCE")
display(field_df)

print("\nINCORRECT PREDICTION EXAMPLES")

for field, values in sorted(field_stats.items()):
    examples = values["incorrect_examples"]

    if examples:
        print(f"\n{field.upper()} ({len(examples)} errors)")
        for example in examples[:3]:
            print({
                "document": example["document"],
                "language": example["language"],
                "expected": example["expected"],
                "predicted": example["predicted"]
            })

FIELD-LEVEL PERFORMANCE


,Field,Correct,Total,Accuracy,Unsupported
3,document_id,26,54,0.481481,13
4,grade,11,18,0.611111,5
5,institution,12,18,0.666667,3
2,date,45,54,0.833333,5
6,name,45,54,0.833333,2
7,phone,15,18,0.833333,2
0,address,32,36,0.888889,0
1,amount,17,18,0.944444,0



INCORRECT PREDICTION EXAMPLES

ADDRESS (4 errors)
{'document': 'kannada_application_080', 'language': 'kannada', 'expected': 'ಬಳ್ಳಾರಿ', 'predicted': 'ಬ'}
{'document': 'kannada_invoice_070', 'language': 'kannada', 'expected': 'ಬಳ್ಳಾರಿ', 'predicted': 'ಸರಕುಪಟ್ಟಿ'}
{'document': 'kannada_application_080', 'language': 'kannada', 'expected': 'ಬಳ್ಳಾರಿ', 'predicted': 'ಅಜಿ೯'}

AMOUNT (1 errors)
{'document': 'kannada_invoice_070', 'language': 'kannada', 'expected': '11090', 'predicted': '11'}

DATE (9 errors)
{'document': 'hindi_application_049', 'language': 'hindi', 'expected': '18/02/2026', 'predicted': '08/02/2026'}
{'document': 'hindi_application_050', 'language': 'hindi', 'expected': '19/02/2026', 'predicted': '09/02/2026'}
{'document': 'hindi_certificate_060', 'language': 'hindi', 'expected': '01/03/2026', 'predicted': '04/03/2026'}

DOCUMENT_ID (28 errors)
{'document': 'hindi_invoice_039', 'language': 'hindi', 'expected': 'INV-0039', 'predicted': '054-0039'}
{'document': 'hindi_invoice_04

In [28]:
import os

history_dir = "/content/drive/MyDrive/BharatDocAI/experiments/finetuning"

for file in os.listdir(history_dir):
    if "history" in file.lower() or "training" in file.lower():
        print(file)

experiment2_training_manifest.csv
experiment2_tokenized_training.json


In [29]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/BharatDocAI"

print("Searching for training history files...\n")

for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        if any(word in file.lower() for word in [
            "history", "loss", "training_log", "train_log"
        ]):
            print(os.path.join(root, file))

Searching for training history files...

/content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/qwen3_lora_training_history.json
/content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_training_history.json


In [30]:
import os
import torch
import json

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated:", round(
    torch.cuda.memory_allocated(0) / 1024**3, 2
), "GB")

print("\nRequired variables:")
for name in [
    "model", "tokenizer", "test_records",
    "unified_records", "CATEGORY_SCHEMAS",
    "SYSTEM_PROMPT", "normalize_evidence"
]:
    print(f"{name}: {'Available' if name in globals() else 'MISSING'}")

print("\nTest records:", len(test_records))

print("\nFirst test record keys:")
print(test_records[0].keys())

print("\nFirst unified record keys:")
print(unified_records[0].keys())

print("\nAdapter directories:")
for path in [
    BASE_CHECKPOINT_DIR,
    EXPERIMENT2_CHECKPOINT_DIR
]:
    print(path, "exists:", os.path.exists(path))
    if os.path.exists(path):
        print("  adapter weights:", os.path.exists(
            os.path.join(path, "adapter_model.safetensors")
        ))
        print("  adapter config:", os.path.exists(
            os.path.join(path, "adapter_config.json")
        ))

GPU: Tesla T4
GPU memory allocated: 3.6 GB

Required variables:
model: Available
tokenizer: Available
test_records: Available
unified_records: Available
CATEGORY_SCHEMAS: Available
SYSTEM_PROMPT: Available
normalize_evidence: Available

Test records: 54

First test record keys:
dict_keys(['document_id', 'language', 'category', 'template_family', 'split', 'ocr_engine', 'corruption', 'severity', 'ocr_text', 'ground_truth', 'ocr_status'])

First unified record keys:
dict_keys(['document_id', 'language', 'category', 'template_family', 'split', 'ocr_engine', 'corruption', 'severity', 'ocr_text', 'ground_truth', 'ocr_status'])

Adapter directories:
/content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora exists: True
  adapter weights: True
  adapter config: True
/content/drive/MyDrive/BharatDocAI/models/checkpoints/qwen3_lora_experiment2 exists: True
  adapter weights: True
  adapter config: True


In [31]:
import pandas as pd

# Only use clean training records
train_clean_records = [
    r for r in unified_records
    if r["split"] == "train"
    and r["corruption"] in [None, "", "clean"]
]

print("Clean training records:", len(train_clean_records))

# Prefer EasyOCR consistently for the demonstrations
demo_records = []

for language in ["english", "hindi", "kannada"]:
    for category in ["invoice", "application", "certificate"]:

        candidates = [
            r for r in train_clean_records
            if r["language"] == language
            and r["category"] == category
            and r["ocr_engine"] == "easyocr"
        ]

        if not candidates:
            # Fallback if EasyOCR is unavailable
            candidates = [
                r for r in train_clean_records
                if r["language"] == language
                and r["category"] == category
            ]

        if not candidates:
            raise ValueError(
                f"No training demonstration found: {language}, {category}"
            )

        # Deterministic selection
        candidates = sorted(candidates, key=lambda x: x["document_id"])
        demo_records.append(candidates[0])

print("\nSelected demonstrations:", len(demo_records))

for r in demo_records:
    print(
        r["language"],
        "|", r["category"],
        "|", r["document_id"],
        "|", r["ocr_engine"]
    )

assert len(demo_records) == 9
assert len(set(r["document_id"] for r in demo_records)) == 9
assert all(r["split"] == "train" for r in demo_records)

print("\nDemonstration integrity checks passed.")

Clean training records: 162

Selected demonstrations: 9
english | invoice | english_invoice_001 | easyocr
english | application | english_application_011 | easyocr
english | certificate | english_certificate_021 | easyocr
hindi | invoice | hindi_invoice_031 | easyocr
hindi | application | hindi_application_041 | easyocr
hindi | certificate | hindi_certificate_051 | easyocr
kannada | invoice | kannada_invoice_061 | easyocr
kannada | application | kannada_application_071 | easyocr
kannada | certificate | kannada_certificate_081 | easyocr

Demonstration integrity checks passed.


In [32]:
from peft import PeftModel
import os

# Detach the currently loaded Experiment 2 adapter
if isinstance(model, PeftModel):
    base_model = model.unload()
else:
    base_model = model

base_model.eval()

print("Base Qwen3-4B restored.")
print("Base model device:", base_model.device)

# Function to attach a selected adapter
def load_lora_adapter(adapter_path):
    global model

    if not os.path.exists(
        os.path.join(adapter_path, "adapter_model.safetensors")
    ):
        raise FileNotFoundError(f"Adapter weights missing: {adapter_path}")

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
        is_trainable=False
    )

    model.eval()
    return model


# Verify Experiment 1
model = load_lora_adapter(BASE_CHECKPOINT_DIR)
print("\nExperiment 1 adapter loaded successfully.")

# Detach Experiment 1
base_model = model.unload()
base_model.eval()

# Verify Experiment 2
model = load_lora_adapter(EXPERIMENT2_CHECKPOINT_DIR)
print("Experiment 2 adapter loaded successfully.")

print("\nBoth adapters verified.")
print("Current model: Experiment 2")

Base Qwen3-4B restored.
Base model device: cuda:0

Experiment 1 adapter loaded successfully.
Experiment 2 adapter loaded successfully.

Both adapters verified.
Current model: Experiment 2


In [33]:
import os
import json
import torch
from tqdm.auto import tqdm
from peft import PeftModel

RESULT_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments/finetuning/four_way_comparison"
)
os.makedirs(RESULT_DIR, exist_ok=True)

# --------------------------------------------------
# 1. JSON parsing
# --------------------------------------------------

def parse_prediction(text):
    text = text.strip()

    if "```" in text:
        text = text.replace("```json", "").replace("```", "").strip()

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        return None

    try:
        result = json.loads(text[start:end + 1])
        return result if isinstance(result, dict) else None
    except json.JSONDecodeError:
        return None


# --------------------------------------------------
# 2. Build prompts
# --------------------------------------------------

def build_user_prompt(record, few_shot=False):
    fields = CATEGORY_SCHEMAS[record["category"]]

    prompt = ""

    if few_shot:
        prompt += "Here are examples of correct information extraction:\n\n"

        for demo in demo_records:
            demo_fields = CATEGORY_SCHEMAS[demo["category"]]

            demo_output = {
                field: demo["ground_truth"].get(field, "")
                for field in demo_fields
            }

            prompt += (
                f"Example language: {demo['language']}\n"
                f"Example category: {demo['category']}\n"
                f"OCR Text:\n{demo['ocr_text']}\n\n"
                f"Correct JSON:\n"
                f"{json.dumps(demo_output, ensure_ascii=False)}\n\n"
            )

    prompt += (
        f"Language: {record['language']}\n"
        f"Category: {record['category']}\n"
        f"OCR Text:\n{record['ocr_text']}\n\n"
        f"Extract these fields: {', '.join(fields)}"
    )

    return prompt


# --------------------------------------------------
# 3. Generate one prediction
# --------------------------------------------------

def generate_prediction(record, few_shot=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": build_user_prompt(record, few_shot)
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    parsed = parse_prediction(response)

    return parsed, response


# --------------------------------------------------
# 4. Run one experimental condition
# --------------------------------------------------

def run_condition(condition_name, adapter_path=None, few_shot=False):

    global model, base_model

    print(f"\nStarting: {condition_name}")

    # Attach the requested model configuration
    if adapter_path is None:
        model = base_model
    else:
        model = PeftModel.from_pretrained(
            base_model,
            adapter_path,
            is_trainable=False
        )

    model.eval()

    results = []

    for record in tqdm(
        test_records,
        desc=condition_name
    ):
        prediction, raw_response = generate_prediction(
            record,
            few_shot=few_shot
        )

        fields = CATEGORY_SCHEMAS[record["category"]]

        # Preserve invalid JSON as an empty prediction object
        prediction_dict = prediction if prediction is not None else {}

        results.append({
            "document_id": record["document_id"],
            "language": record["language"],
            "category": record["category"],
            "template_family": record["template_family"],
            "ocr_engine": record["ocr_engine"],
            "corruption": record["corruption"],
            "severity": record["severity"],
            "ocr_text": record["ocr_text"],
            "ground_truth": record["ground_truth"],
            "prediction": prediction_dict,
            "raw_response": raw_response,
            "json_valid": prediction is not None,
            "schema_valid": (
                prediction is not None
                and set(prediction.keys()) == set(fields)
            )
        })

    # Save results
    output_path = os.path.join(
        RESULT_DIR,
        f"{condition_name}_predictions.jsonl"
    )

    with open(output_path, "w", encoding="utf-8") as f:
        for result in results:
            f.write(
                json.dumps(result, ensure_ascii=False) + "\n"
            )

    print(f"\nCompleted: {condition_name}")
    print("Predictions:", len(results))
    print("Valid JSON:", sum(r["json_valid"] for r in results))
    print("Schema valid:", sum(r["schema_valid"] for r in results))
    print("Saved:", output_path)

    # Detach adapter to restore the shared base model
    if adapter_path is not None:
        base_model = model.unload()
        base_model.eval()
    else:
        model = None

    return results


# --------------------------------------------------
# 5. Run all four conditions
# --------------------------------------------------

comparison_results = {}

comparison_results["zero_shot"] = run_condition(
    "zero_shot",
    adapter_path=None,
    few_shot=False
)

comparison_results["few_shot"] = run_condition(
    "few_shot",
    adapter_path=None,
    few_shot=True
)

comparison_results["experiment1_lora"] = run_condition(
    "experiment1_lora",
    adapter_path=BASE_CHECKPOINT_DIR,
    few_shot=False
)

comparison_results["experiment2_lora"] = run_condition(
    "experiment2_lora",
    adapter_path=EXPERIMENT2_CHECKPOINT_DIR,
    few_shot=False
)

print("\n" + "=" * 55)
print("FOUR-WAY COMPARISON INFERENCE COMPLETED")
print("=" * 55)

for name, results in comparison_results.items():
    print(name, ":", len(results), "predictions")

print("\nResults directory:", RESULT_DIR)


Starting: zero_shot


zero_shot:   0%|          | 0/54 [00:00<?, ?it/s]


Completed: zero_shot
Predictions: 54
Valid JSON: 54
Schema valid: 54
Saved: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/zero_shot_predictions.jsonl

Starting: few_shot


few_shot:   0%|          | 0/54 [00:00<?, ?it/s]


Completed: few_shot
Predictions: 54
Valid JSON: 54
Schema valid: 54
Saved: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/few_shot_predictions.jsonl

Starting: experiment1_lora


experiment1_lora:   0%|          | 0/54 [00:00<?, ?it/s]


Completed: experiment1_lora
Predictions: 54
Valid JSON: 54
Schema valid: 54
Saved: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/experiment1_lora_predictions.jsonl

Starting: experiment2_lora


experiment2_lora:   0%|          | 0/54 [00:00<?, ?it/s]


Completed: experiment2_lora
Predictions: 54
Valid JSON: 54
Schema valid: 54
Saved: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/experiment2_lora_predictions.jsonl

FOUR-WAY COMPARISON INFERENCE COMPLETED
zero_shot : 54 predictions
few_shot : 54 predictions
experiment1_lora : 54 predictions
experiment2_lora : 54 predictions

Results directory: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

RESULT_DIR = (
    "/content/drive/MyDrive/BharatDocAI/"
    "experiments/finetuning/four_way_comparison"
)

print("Folder exists:", os.path.exists(RESULT_DIR))

if os.path.exists(RESULT_DIR):
    print("\nSaved prediction files:")
    for file in sorted(os.listdir(RESULT_DIR)):
        print(file)

Folder exists: True

Saved prediction files:
experiment1_lora_predictions.jsonl
experiment2_lora_predictions.jsonl
few_shot_predictions.jsonl
zero_shot_predictions.jsonl


In [5]:
import json

paths = {
    "zero_shot": "/content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/zero_shot_predictions.jsonl",
    "experiment1": "/content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/experiment1_lora_predictions.jsonl",
    "experiment2": "/content/drive/MyDrive/BharatDocAI/experiments/finetuning/four_way_comparison/experiment2_lora_predictions.jsonl"
}

results = {}

for name, path in paths.items():
    with open(path, encoding="utf-8") as f:
        results[name] = [json.loads(line) for line in f]

for name in ["experiment1", "experiment2"]:
    same_predictions = sum(
        results["zero_shot"][i]["prediction"]
        == results[name][i]["prediction"]
        for i in range(54)
    )

    same_raw = sum(
        results["zero_shot"][i]["raw_response"]
        == results[name][i]["raw_response"]
        for i in range(54)
    )

    print(f"\nZero-shot vs {name}")
    print("Identical predictions:", same_predictions, "/ 54")
    print("Identical raw responses:", same_raw, "/ 54")


Zero-shot vs experiment1
Identical predictions: 17 / 54
Identical raw responses: 0 / 54

Zero-shot vs experiment2
Identical predictions: 54 / 54
Identical raw responses: 54 / 54


In [6]:
import os
import json
import hashlib

PROJECT_ROOT = "/content/drive/MyDrive/BharatDocAI"

EXP1_DIR = os.path.join(
    PROJECT_ROOT, "models/checkpoints/qwen3_lora"
)

EXP2_DIR = os.path.join(
    PROJECT_ROOT, "models/checkpoints/qwen3_lora_experiment2"
)

def inspect_adapter(path, name):
    print(f"\n{name}")
    print("-" * 45)

    for filename in ["adapter_config.json", "adapter_model.safetensors"]:
        filepath = os.path.join(path, filename)

        print(f"{filename}:")
        print("  Exists:", os.path.exists(filepath))

        if os.path.exists(filepath):
            print("  Size:", round(os.path.getsize(filepath) / (1024**2), 2), "MB")

            with open(filepath, "rb") as f:
                digest = hashlib.sha256(f.read()).hexdigest()

            print("  SHA256:", digest)

    config_path = os.path.join(path, "adapter_config.json")

    if os.path.exists(config_path):
        with open(config_path, encoding="utf-8") as f:
            config = json.load(f)

        print("\nAdapter configuration:")
        for key in ["base_model_name_or_path", "r", "lora_alpha", "target_modules"]:
            print(f"  {key}: {config.get(key)}")

inspect_adapter(EXP1_DIR, "EXPERIMENT 1")
inspect_adapter(EXP2_DIR, "EXPERIMENT 2")


EXPERIMENT 1
---------------------------------------------
adapter_config.json:
  Exists: True
  Size: 0.0 MB
  SHA256: ce4b2f90f3ac213720e978e44b414e65988ec19b824e9cfa919eb8c198b25deb
adapter_model.safetensors:
  Exists: True
  Size: 63.06 MB
  SHA256: 431efc94c467dde9a92b45aa00b1e9c3fbd1ecf61231b0efcb5916a663a99f3a

Adapter configuration:
  base_model_name_or_path: Qwen/Qwen3-4B
  r: 8
  lora_alpha: 16
  target_modules: ['down_proj', 'up_proj', 'k_proj', 'o_proj', 'gate_proj', 'v_proj', 'q_proj']

EXPERIMENT 2
---------------------------------------------
adapter_config.json:
  Exists: True
  Size: 0.0 MB
  SHA256: 14cd7141289fda05ee39c4d0909c910a3dae7debb255add4446cf0cbea0284d0
adapter_model.safetensors:
  Exists: True
  Size: 63.06 MB
  SHA256: 74bcd93be4b65871a48c13d37c0abbd12efbbd2813a77ff8e3b31e2cac3ccca5

Adapter configuration:
  base_model_name_or_path: Qwen/Qwen3-4B
  r: 8
  lora_alpha: 16
  target_modules: ['q_proj', 'gate_proj', 'up_proj', 'k_proj', 'v_proj', 'o_proj', 'do

In [7]:
import os
import json
import torch
from safetensors import safe_open

def inspect_weights(path, name):
    config_path = os.path.join(path, "adapter_config.json")
    weights_path = os.path.join(path, "adapter_model.safetensors")

    print(f"\n{'='*55}")
    print(name)
    print("="*55)

    with open(config_path, encoding="utf-8") as f:
        config = json.load(f)

    print("\nFull adapter configuration:")
    print(json.dumps(config, indent=2))

    tensor_stats = {}

    with safe_open(weights_path, framework="pt", device="cpu") as f:
        keys = list(f.keys())
        print("\nNumber of adapter tensors:", len(keys))

        for key in keys[:8]:
            tensor = f.get_tensor(key).float()
            tensor_stats[key] = {
                "shape": tuple(tensor.shape),
                "mean": tensor.mean().item(),
                "std": tensor.std().item(),
                "norm": tensor.norm().item()
            }

    print("\nFirst 8 tensor statistics:")
    for key, stats in tensor_stats.items():
        print(key, stats)

    return config, tensor_stats

exp1_config, exp1_stats = inspect_weights(
    EXP1_DIR, "EXPERIMENT 1"
)

exp2_config, exp2_stats = inspect_weights(
    EXP2_DIR, "EXPERIMENT 2"
)


EXPERIMENT 1

Full adapter configuration:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "Qwen/Qwen3-4B",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "kasa_config": null,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 16,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.21.2",
  "qalora_group_size": 16,
  "r": 8,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "down_proj",
    "up_proj",
    "k_proj",
    "o_proj",
    "gate_proj",
    "v_proj",
    "q_proj"


In [8]:
from safetensors import safe_open
import torch
import os

exp1_weights = os.path.join(EXP1_DIR, "adapter_model.safetensors")
exp2_weights = os.path.join(EXP2_DIR, "adapter_model.safetensors")

with safe_open(exp1_weights, framework="pt", device="cpu") as f1, \
     safe_open(exp2_weights, framework="pt", device="cpu") as f2:

    keys1 = set(f1.keys())
    keys2 = set(f2.keys())

    print("Same tensor keys:", keys1 == keys2)
    print("Experiment 1 tensors:", len(keys1))
    print("Experiment 2 tensors:", len(keys2))

    differences = []

    for key in sorted(keys1 & keys2):
        w1 = f1.get_tensor(key).float()
        w2 = f2.get_tensor(key).float()

        diff = (w1 - w2).abs()

        differences.append({
            "key": key,
            "mean_abs_diff": diff.mean().item(),
            "max_abs_diff": diff.max().item(),
            "exp1_norm": w1.norm().item(),
            "exp2_norm": w2.norm().item()
        })

    changed = sum(x["mean_abs_diff"] > 0 for x in differences)

    print("\nChanged tensors:", changed, "/", len(differences))

    print("\nFirst 10 tensor differences:")
    for item in differences[:10]:
        print(item)

    print("\nAverage mean absolute difference:",
          sum(x["mean_abs_diff"] for x in differences) / len(differences))

Same tensor keys: True
Experiment 1 tensors: 504
Experiment 2 tensors: 504

Changed tensors: 504 / 504

First 10 tensor differences:
{'key': 'base_model.model.model.layers.0.mlp.down_proj.lora_A.weight', 'mean_abs_diff': 0.006936186458915472, 'max_abs_diff': 0.024620484560728073, 'exp1_norm': 1.673858880996704, 'exp2_norm': 1.6740511655807495}
{'key': 'base_model.model.model.layers.0.mlp.down_proj.lora_B.weight', 'mean_abs_diff': 0.0014038693625479937, 'max_abs_diff': 0.00626444723457098, 'exp1_norm': 0.17612802982330322, 'exp2_norm': 0.17934751510620117}
{'key': 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.weight', 'mean_abs_diff': 0.013335073366761208, 'max_abs_diff': 0.04179174453020096, 'exp1_norm': 1.6432383060455322, 'exp2_norm': 1.6437841653823853}
{'key': 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.weight', 'mean_abs_diff': 0.0013765996554866433, 'max_abs_diff': 0.006423581391572952, 'exp1_norm': 0.33731284737586975, 'exp2_norm': 0.3401617705821991}
{'key': '

In [9]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Free GPU memory:",
          round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

CUDA available: False
